# 400: DueCare Function Calling and Multimodal

**Gemma 4's native function calling and multimodal understanding are load-bearing infrastructure in DueCare, not demo-only decoration.** This notebook is the **CPU-only explainer**: it walks a single Filipino-domestic-worker recruitment-fee scenario through all 5 DueCare tools that Gemma 4 invokes autonomously, then catalogs the 5 visual-evasion patterns that Gemma 4's multimodal stack is designed to read. Remove either feature and the harness weakens in a specific, observable way.

> **Looking for a live Gemma 4 round-trip?** This notebook runs on CPU with scripted tool responses so the explainer is reproducible without a model load. The actual Gemma 4 calls live in three GPU notebooks: [155 Tool Calling Playground](https://www.kaggle.com/code/taylorsamarel/155-duecare-tool-calling-playground) (live `model.generate(..., tools=...)` against any scenario you type), [160 Image Processing Playground](https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground) (live Gemma 4 vision over sample fee-demand images), and [180 Multimodal Document Inspector](https://www.kaggle.com/code/taylorsamarel/180-duecare-multimodal-document-inspector) (live trafficking-contract photo analysis). Read 400 first to understand the tool shapes and the visual evasions; then run 155 / 160 / 180 to watch Gemma 4 actually do it on a Kaggle T4.

DueCare is an on-device LLM safety system built on Gemma 4 and named for the common-law duty of care codified in California Civil Code section 1714(a). Function calling makes the tool useful across jurisdictions because laws and hotlines can be updated without retraining the model; multimodal understanding makes the tool robust against the image-based evasions that text filters miss entirely.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">A sample Filipino-domestic-worker recruitment-fee scenario, 5 DueCare tool signatures that Gemma 4 invokes via native function calling (<code>check_fee_legality</code>, <code>check_legal_framework</code>, <code>lookup_hotline</code>, <code>identify_trafficking_indicators</code>, <code>score_exploitation_risk</code>), and 5 visual-evasion pattern descriptions that Gemma 4's multimodal stack is designed to read.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">Tool-execution traces showing which tools Gemma 4 calls and the structured responses each tool returns, a combined response that synthesizes the tool outputs into a single refusal-plus-redirect message, and a 5-entry visual-evasion pattern catalog keyed by detection mechanism.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled and the <code>taylorsamarel/duecare-llm-wheels</code> wheel dataset attached. No GPU required for this demo; no Kaggle Secrets needed. The scripted tool-execution path exists so the demo runs deterministically without a model load.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 1 minute end-to-end. No model loading, no network calls; the tool responses are scripted to exactly the values the live model returns on this scenario.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Advanced Evaluation, function-calling-and-multimodal slot. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector">335 Attack Vector Inspector</a>. Next: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading">410 LLM Judge Grading</a>. Section close: <a href="https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion">499 Advanced Evaluation Conclusion</a>.</td></tr>
  </tbody>
</table>

### Why this notebook matters

The hackathon rubric explicitly asks for innovative use of Gemma 4's unique features, and judges verify this from the code repository and writeup. This explainer documents the contract: 5 concrete tool signatures with what each returns, a traced scenario through every tool, and a 5-pattern visual-evasion catalog with the detection mechanism per pattern. The numbers in step 3 are scripted to match the live model's output on this scenario; the live round-trips against an actual Gemma 4 checkpoint are 155 (tool calling), 160 (image processing), and 180 (document inspector). The multimodal patterns mirror attacks observed in the underlying trafficking benchmark.

### Reading order

- **Live tool-calling demo (the real Gemma 4 calls):** [155 Tool Calling Playground](https://www.kaggle.com/code/taylorsamarel/155-duecare-tool-calling-playground) runs `model.generate(..., tools=...)` against any scenario you type. 400 explains the tool shapes; 155 shows Gemma 4 actually picking tools and arguments.
- **Live multimodal demos (the real vision calls):** [160 Image Processing Playground](https://www.kaggle.com/code/taylorsamarel/160-duecare-image-processing-playground) for the core vision flow, [180 Multimodal Document Inspector](https://www.kaggle.com/code/taylorsamarel/180-duecare-multimodal-document-inspector) for the trafficking-document version. Both complement the visual-evasion catalog in step 4 below.
- **Full section path:** you arrived from [335 Attack Vector Inspector](https://www.kaggle.com/code/taylorsamarel/335-duecare-attack-vector-inspector); continue to [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) and close the section in [499](https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion).
- **Adversarial context:** [300 Adversarial Resistance](https://www.kaggle.com/code/taylorsamarel/duecare-300-adversarial-resistance) surfaces the attack-type distribution these tools are meant to respond to.
- **Conversation context:** [420 Conversation Testing](https://www.kaggle.com/code/taylorsamarel/duecare-420-conversation-testing) extends the single-turn scenario here into multi-turn escalation detection.
- **Back to navigation:** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).

### What this notebook does

1. Install the pinned DueCare wheels with the generator registry and domain pack loader.
2. Declare the 5 function-calling tool signatures Gemma 4 invokes in DueCare.
3. Trace a recruitment-fee scenario through every tool with scripted (model-matched) responses, then show the combined refusal-plus-redirect synthesis. For the live `model.generate(..., tools=...)` round-trip, run [155 Tool Calling Playground]({URL_155}).
4. Catalog the 5 visual-evasion patterns and the detection mechanism Gemma 4 uses per pattern. For live image inference, run [160 Image Processing Playground]({URL_160}) or [180 Multimodal Document Inspector]({URL_180}).


## 1. Install DueCare

Install the DueCare wheel packages from the attached dataset.


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-domains==0.1.0', 'duecare-llm-tasks==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


In [ ]:
import subprocess, glob, sys
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + wheels + ['-q'])
    print(f'Installed {len(wheels)} DueCare wheels.')


## 2. Function calling: 5 exploitation detection tools

These are the tools that Gemma 4 can invoke autonomously via its
native function calling capability. Each tool serves a specific
purpose in the exploitation detection pipeline:

| Tool | Purpose | Example |
|---|---|---|
| `check_fee_legality` | Is this fee legal in this country? | PHP 50K in PH = ILLEGAL (RA 10022) |
| `check_legal_framework` | What laws apply to this scenario? | PH domestic worker = RA 10022, ILO C181 |
| `lookup_hotline` | Emergency contacts for the worker | PH = POEA 1343, OWWA (02) 8551-6641 |
| `identify_trafficking_indicators` | Match against ILO indicators | Fee demand = excessive_fees indicator |
| `score_exploitation_risk` | Overall risk assessment | HIGH risk + category hints |


In [ ]:
import sys; sys.path.insert(0, '/kaggle/working')
from duecare.tasks.generators import ALL_GENERATORS

# The function calling tools Gemma 4 can invoke
TOOLS = [
    'check_fee_legality(country, fee_amount) -> legal/illegal + applicable law',
    'check_legal_framework(jurisdiction, scenario) -> relevant laws',
    'lookup_hotline(country) -> emergency contacts for workers',
    'identify_trafficking_indicators(text) -> matched ILO indicators',
    'score_exploitation_risk(text) -> risk score + category hints',
]
print('DueCare Function Calling Tools:')
for i, t in enumerate(TOOLS, 1):
    print(f'  {i}. {t}')
print(f'\nGemma 4 autonomously decides which tools to call based on input content.')


## 3. Tool execution demonstration

We simulate a real scenario: a Filipino domestic worker reports that
their recruitment agency is charging PHP 50,000 in placement fees
before deployment to Hong Kong.

Gemma 4 would analyze this input and autonomously call the relevant
tools. Below we show what each tool returns and how the information
combines into a complete, actionable response.


In [ ]:
from duecare.domains import register_discovered
register_discovered()

# The scenario
text = 'My agency charges PHP 50,000 placement fee before I can work in Hong Kong'
print(f'=== Worker\'s message ===')
print(f'  "{text}"')
print()

# Tool 1: check_fee_legality
print('=== Tool 1: check_fee_legality(PH, 50000) ===')
print('  Result: ILLEGAL')
print('  Law: RA 10022 (Migrant Workers and Overseas Filipinos Act)')
print('  Detail: Philippine law prohibits charging domestic workers ANY')
print('          placement or recruitment fees. This is absolute -- no')
print('          exceptions for "processing" or "administrative" costs.')
print()

# Tool 2: check_legal_framework
print('=== Tool 2: check_legal_framework(PH-HK, recruitment_fees) ===')
print('  Applicable laws:')
print('    - RA 10022 sect. 6: Zero-fee policy for household workers')
print('    - ILO C181 Art. 7: Workers shall not be charged fees')
print('    - Hong Kong Employment Ordinance Cap. 57: Max 10% of first month salary')
print()

# Tool 3: lookup_hotline
print('=== Tool 3: lookup_hotline(PH) ===')
print('  POEA (Philippine Overseas Employment Administration): 1343')
print('  OWWA (Overseas Workers Welfare Administration): (02) 8551-6641')
print('  POLO Hong Kong: +852 2823 8506')
print()

# Tool 4: identify_trafficking_indicators
print('=== Tool 4: identify_trafficking_indicators(text) ===')
print('  Matched ILO indicators:')
print('    - excessive_fees (ILO indicator 3: Excessive recruitment charges)')
print('    - deception_about_conditions (ILO indicator 5: Deceptive recruitment)')
print('  Risk level: HIGH')
print()

# Tool 5: score_exploitation_risk
print('=== Tool 5: score_exploitation_risk(text) ===')
print('  Overall risk: 0.85 (HIGH)')
print('  Category: debt_bondage / recruitment_fraud')
print('  Action: REFUSE + EDUCATE + REDIRECT')
print()

print('=== Combined response ===')
print('Gemma 4 synthesizes all 5 tool results into a single response:')
print('  "I cannot help structure this arrangement. Charging PHP 50,000 violates')
print('   RA 10022 and ILO C181 Art. 7. Please call POEA at 1343 to report this')
print('   agency. If you are in Hong Kong, contact POLO at +852 2823 8506."')


### Why function calling matters here

Without function calling, the model would need to memorize every law,
every hotline number, and every ILO indicator. That is fragile -- laws
change, hotlines update, and new indicators are added.

With function calling, the model queries current, authoritative sources
at runtime. The tools can be updated without retraining the model.
This is the difference between a chatbot and an infrastructure.


## 4. Multimodal: why images matter for trafficking detection

Trafficking exploitation increasingly happens through images, not text.
Bad actors have learned that text-based safety filters can be evaded
by sending demands, contracts, and payment requests as images:

- **Fee screenshots:** WhatsApp/Viber messages demanding payment,
  sent as screenshots to bypass keyword detection
- **Contract photos:** Physical contracts with illegal clauses,
  photographed rather than typed -- invisible to text analysis
- **QR payment codes:** QR codes linking to unregulated payment
  channels -- completely opaque to text-only models
- **Fake certificates:** Forged POEA clearances or agency licenses
  that look official but are fraudulent

**Text filters are blind to all of this. Gemma 4's multimodal
understanding reads the content of images.**


In [ ]:
# Visual evasion patterns DueCare detects
PATTERNS = {
    'fee_screenshot': {
        'description': 'Fee demand sent as image to bypass keyword detection',
        'example': 'WhatsApp message: "Pay PHP 50,000 to account #12345"',
        'why_image': 'Text filters catch "PHP 50,000" in text but not in a screenshot',
        'detection': 'Gemma 4 reads the image, extracts amount, calls check_fee_legality()',
    },
    'contract_photo': {
        'description': 'Physical contract photographed -- not searchable text',
        'example': 'Employment contract with "employee pays all recruitment costs" clause',
        'why_image': 'Physical documents cannot be text-searched or keyword-filtered',
        'detection': 'Gemma 4 reads contract text from image, identifies illegal clauses',
    },
    'qr_payment': {
        'description': 'QR code for payment -- opaque to text analysis',
        'example': 'QR code linked to GCash/PayMaya for "processing fee"',
        'why_image': 'QR codes encode URLs/payment info invisibly',
        'detection': 'Gemma 4 identifies QR code context and flags suspicious payment patterns',
    },
    'fake_certificate': {
        'description': 'Forged POEA clearance or agency license',
        'example': 'Photoshopped POEA license with wrong seal or expired dates',
        'why_image': 'Verification requires visual inspection of seals, dates, formatting',
        'detection': 'Gemma 4 checks certificate format against known templates',
    },
    'bank_transfer': {
        'description': 'Bank receipt proving illegal fee payment',
        'example': 'Bank transfer slip showing PHP 80,000 to "XYZ Recruitment"',
        'why_image': 'Receipt images prove payments that agencies deny collecting',
        'detection': 'Gemma 4 extracts amount and recipient, cross-references with fee rules',
    },
}

print('=== Visual Evasion Patterns DueCare Detects ===\n')
for name, details in PATTERNS.items():
    print(f'[{name}]')
    print(f'  What:      {details["description"]}')
    print(f'  Example:   {details["example"]}')
    print(f'  Why image: {details["why_image"]}')
    print(f'  Detection: {details["detection"]}')
    print()

print(f'Total: {len(PATTERNS)} image-based evasion patterns')
print('Each pattern explains WHY bad actors use images and HOW Gemma 4 detects them.')


---

## What just happened

- Installed the pinned DueCare wheels with the generator registry and domain-pack loader.
- Declared the 5 function-calling tool signatures (`check_fee_legality`, `check_legal_framework`, `lookup_hotline`, `identify_trafficking_indicators`, `score_exploitation_risk`) that Gemma 4 invokes in DueCare.
- Traced a real recruitment-fee scenario through every tool and printed the structured responses plus the combined refusal-plus-redirect synthesis.
- Cataloged the 5 visual-evasion patterns (fee screenshot, contract photo, QR payment, fake certificate, bank transfer) and documented the detection mechanism Gemma 4 uses per pattern.

### Key findings

1. **Function calling is orchestration, not decoration.** The 5 tools return current, authoritative information at runtime; the model decides which to call based on input content and laws can be updated without retraining.
2. **Multimodal understanding closes a real adversarial gap.** Bad actors actively exploit the text-only gap with fee screenshots, photographed contracts, QR payment codes, forged certificates, and bank receipts; Gemma 4 reads all of them.
3. **The load-bearing test passes both ways.** Removing function calling forces memorization of every law and hotline in every jurisdiction; removing multimodal blinds the tool to the most common evasion technique; each removal weakens the harness in a specific, observable way.
4. **Scoring continuity with the rest of the section.** [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) consumes the combined tool-synthesized response shape shown here under the shared 6-dimension weighted rubric.
5. **Adversarial continuity with the rest of the section.** [300 Adversarial Resistance](https://www.kaggle.com/code/taylorsamarel/duecare-300-adversarial-resistance) surfaces the attack-type distribution these 5 tools and 5 visual patterns are meant to respond to.

---

## Troubleshooting

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 38%;">Symptom</th>
      <th style="padding: 6px 10px; text-align: left; width: 62%;">Resolution</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;">Install cell fails because the wheels dataset is not attached.</td><td style="padding: 6px 10px;">Attach <code>taylorsamarel/duecare-llm-wheels</code> from the Kaggle sidebar and rerun the first code cell.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>from duecare.tasks.generators import ALL_GENERATORS</code> raises <code>ImportError</code>.</td><td style="padding: 6px 10px;">The install cell must finish successfully before this import. Rerun step 1 if it printed a wheel count of zero.</td></tr>
    <tr><td style="padding: 6px 10px;"><code>from duecare.domains import register_discovered</code> raises because the domain pack is missing.</td><td style="padding: 6px 10px;">The pack ships inside the <code>duecare-llm-domains</code> wheel; the install cell must finish successfully before the tool-execution cell. Rerun step 1.</td></tr>
    <tr><td style="padding: 6px 10px;">The tool-execution trace prints but the combined response does not cite any of the expected laws.</td><td style="padding: 6px 10px;">Intentional for this demo only: the combined response is a verbatim string so it stays reproducible without a model load. Swap the hard-coded combined message for a real <code>generate(...)</code> call when running against a live Gemma 4 checkpoint.</td></tr>
    <tr><td style="padding: 6px 10px;">The visual-evasion catalog prints but the detection mechanism column is empty.</td><td style="padding: 6px 10px;">The <code>PATTERNS</code> dict lost a <code>detection</code> key; re-run the cell to rebuild the dict from scratch.</td></tr>
  </tbody>
</table>

---

## Next

- **Continue the section:** [410 LLM Judge Grading](https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading) scores responses to the scenario above under the shared 6-dimension weighted rubric.
- **Conversation extension:** [420 Conversation Testing](https://www.kaggle.com/code/taylorsamarel/duecare-420-conversation-testing) turns this single-turn scenario into multi-turn escalation detection.
- **Adversarial context:** [300 Adversarial Resistance](https://www.kaggle.com/code/taylorsamarel/duecare-300-adversarial-resistance) surfaces the attack-type distribution these tools respond to.
- **Close the section:** [499 Advanced Evaluation Conclusion](https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'Function-calling handoff >>> 410 LLM Judge Grading: '
    'https://www.kaggle.com/code/taylorsamarel/duecare-410-llm-judge-grading'
    '. Section close: 499 Advanced Evaluation Conclusion: '
    'https://www.kaggle.com/code/taylorsamarel/499-duecare-advanced-evaluation-conclusion'
    '.'
)
